# Ch 13 · Lab 1 — Sonar 베이스라인 (분리 없음)

원본: `04-Sonar.py`

학습/검증 분리 없이 *전체로 학습*. **베이스라인의 함정**을 직접 확인하기 위한 단계.

## 0. 환경

In [1]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import Input, Sequential
from keras.layers import Dense

keras.utils.set_random_seed(0)

print("Keras:", keras.__version__)

from sklearn.preprocessing import LabelEncoder

2026-05-12 16:38:45.218311: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778571525.234223 1503347 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778571525.241595 1503347 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Keras: 3.14.1


## 1. 데이터

In [2]:
DATA = "../../data/sonar.csv"
df = pd.read_csv(DATA, header=None)
X = df.iloc[:, 0:60].to_numpy(dtype="float32")
y_str = df.iloc[:, 60].to_numpy()
y = LabelEncoder().fit_transform(y_str).astype("float32")
print("X:", X.shape, "y:", y.shape, "분포:", np.bincount(y.astype(int)))

X: (208, 60) y: (208,) 분포: [111  97]


## 2. 모델

In [3]:
def build_model():
    return Sequential([
        Input(shape=(60,)),
        Dense(24, activation="relu"),
        Dense(10, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

keras.utils.set_random_seed(0)
model = build_model()
model.compile(loss="mean_squared_error", optimizer="adam", metrics=["accuracy"])
model.summary()

2026-05-12 16:38:47.448786: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 24)             │         1,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,725 (6.74 KB)

 Trainable params: 1,725 (6.74 KB)

 Non-trainable params: 0 (0.00 B)

## 3. 학습 (전체 데이터로)

In [4]:
hist = model.fit(X, y, epochs=200, batch_size=5, verbose=0)
print(f"final train accuracy: {hist.history['accuracy'][-1]:.4f}")

final train accuracy: 1.0000


## 4. 같은 데이터로 평가 — 함정

In [5]:
loss, acc = model.evaluate(X, y, verbose=0)
print(f"전체 데이터 정확도: {acc:.4f}")

전체 데이터 정확도: 1.0000


> 학습에 쓴 데이터로 평가하면 정확도가 매우 높게 나옴. **일반화 성능을 모른다**는 것이 문제. → Lab 2에서 train/test 분리 도입.